In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import json
from datetime import datetime

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_FILE  = 'final_merged_data.xlsx'
MARKETS    = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY  = 96          # 96 × 15 min = 24 hours
RANDOM_SEED = 42

# Train/Val/Test split dates
TRAIN_END = '2024-12-31'
VAL_END   = '2025-09-30'
# Test = everything after VAL_END

np.random.seed(RANDOM_SEED)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 1 — LOAD & FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1: Loading and transforming data")
print("=" * 60)

sheets = pd.read_excel(DATA_FILE, sheet_name=None)


def build_features(df):
    """
    Full feature engineering pipeline.
    Input : raw sheet dataframe
    Output: enriched dataframe with all 13 model features
    """
    df = df.copy()

    # ── Datetime ──────────────────────────────────────────────
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    # ── Drop redundant/collinear columns ──────────────────────
    # apparent_temperature  → r=0.97 with temperature_2m  → DROP
    # sunshine_duration     → r=0.93 with shortwave_radiation → DROP
    # precipitation         → r=0.99 with rain              → DROP
    df = df.drop(columns=[
        'apparent_temperature (°C)',
        'sunshine_duration (s)',
        'precipitation (mm)'
    ], errors='ignore')

    # ── A) Temporal features ──────────────────────────────────
    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek   # 0=Mon
    df['month'] = df['datetime'].dt.month

    # Cyclical encodings (sin/cos so model knows 23h is close to 0h)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    # Holiday flag (Indian national holidays)
    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        # If holidays library not installed, use zeros
        df['holiday_flag'] = 0
        print("  [Note] pip install holidays for holiday_flag feature")

    # ── B) Autoregressive features ────────────────────────────
    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)    # same slot yesterday
    df['price_lag672'] = p.shift(672)   # same slot last week
    df['ma3d']  = p.shift(1).rolling(288).mean()   # 3-day moving avg
    df['ma7d']  = p.shift(1).rolling(672).mean()   # 7-day moving avg

    # ── C) Weather / supply-side features ─────────────────────
    # temperature quadratic (U-shaped demand effect)
    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2

    # solar radiation (strongest weather signal, r=−0.36)
    df['radiation'] = df['shortwave_radiation (W/m²)']

    # interaction: solar × hour_sin captures duck curve explicitly
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']

    # wind speed at 100m (turbine hub height proxy)
    df['wind_speed'] = df['wind_speed_100m (km/h)']

    # cloud cover
    df['cloud_cover'] = df['cloud_cover (%)']

    # humidity (monsoon proxy)
    df['humidity'] = df['relative_humidity_2m (%)']

    # rain (keep one precipitation proxy)
    df['rain'] = df['rain (mm)']

    # ── Log-price target (compresses spike tail) ──────────────
    df['log_price'] = np.log1p(df['Price (Rs./MWh)'])

    return df


# Build features for each market
market_dfs = {}
for mkt in MARKETS:
    print(f"  Building features for {mkt}...")
    market_dfs[mkt] = build_features(sheets[mkt])
    print(f"    Shape: {market_dfs[mkt].shape}  |  "
          f"Date range: {market_dfs[mkt]['datetime'].min().date()} → "
          f"{market_dfs[mkt]['datetime'].max().date()}")

# ── Feature lists used in each model ─────────────────────────────
# (P(t-1) and P(t-4) EXCLUDED — not available 24h ahead)
LGBM_FEATURES = [
    # Temporal
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    # Autoregressive
    'price_lag96', 'price_lag672',
    'ma3d', 'ma7d',
    # Weather
    'radiation', 'radiation_x_hour_sin',
    'temp', 'temp_sq',
    'wind_speed', 'cloud_cover',
    'humidity', 'rain',
]

print(f"\nLightGBM features    : {len(LGBM_FEATURES)}")

STEP 1: Loading and transforming data
  Building features for GDAM...
    Shape: (144864, 36)  |  Date range: 2022-01-12 → 2026-02-28
  Building features for DAM...
    Shape: (144864, 36)  |  Date range: 2022-01-12 → 2026-02-28
  Building features for RTM...
    Shape: (144116, 36)  |  Date range: 2022-01-12 → 2026-02-28

LightGBM features    : 20


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Train/Val/Test split and scaling")
print("=" * 60)

from sklearn.preprocessing import StandardScaler


def split_and_scale(df, feature_cols):
    """
    Time-based split. NEVER shuffle — future must not leak into train.
    Returns split dataframes and fitted scaler.
    """
    df = df.dropna(subset=feature_cols + ['log_price']).copy()

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]

    print(f"    Train: {len(train):,} rows  |  "
          f"Val: {len(val):,}  |  Test: {len(test):,}")

    # Fit scaler on TRAIN only — never on val/test
    scaler = StandardScaler()
    train_scaled = train.copy()
    val_scaled   = val.copy()
    test_scaled  = test.copy()

    train_scaled[feature_cols] = scaler.fit_transform(train[feature_cols])
    val_scaled[feature_cols]   = scaler.transform(val[feature_cols])
    test_scaled[feature_cols]  = scaler.transform(test[feature_cols])

    return train_scaled, val_scaled, test_scaled, scaler, train, val, test


# ══════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 100
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:30s}  MAE={mae:.1f}  RMSE={rmse:.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def plot_predictions(y_true, y_pred, title, path, n_days=3):
    """Plot actual vs predicted for the first n_days of test set."""
    n = n_days * SLOTS_DAY
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')

    # Full test set
    axes[0].plot(y_true[:500], color='#aaa', lw=0.8, label='Actual', alpha=0.8)
    axes[0].plot(y_pred[:500], color='#e74c3c', lw=0.8, label='Predicted', alpha=0.8)
    axes[0].set_title(f'{title} — First 500 slots of Test Set',
                      color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].tick_params(colors='#aaa')
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')

    # Zoomed: first 3 days
    axes[1].plot(y_true[:n], color='#aaa', lw=1.2, label='Actual')
    axes[1].plot(y_pred[:n], color='#27ae60', lw=1.2, label='Predicted', ls='--')
    axes[1].set_title(f'{title} — First {n_days} Days (zoomed)',
                      color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].tick_params(colors='#aaa')
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[1].set_xlabel('Slot (15 min each)', color='#aaa')

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()



STEP 2: Train/Val/Test split and scaling


In [ ]:
import os

# Create the outputs directory if it doesn't exist
if not os.path.exists('outputs'):
    os.makedirs('outputs')
    print("Created 'outputs/' directory.")

Created 'outputs/' directory.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 1 — LightGBM
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 1: LightGBM — 24h-Ahead Price Forecasting")
print("=" * 60)

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    print("  [SKIP] pip install lightgbm")
    LGBM_AVAILABLE = False

all_metrics = {'lgbm': {}, 'cnnlstm': {}, 'tft': {}}

if LGBM_AVAILABLE:
    for mkt in MARKETS:
        print(f"\n  Training LightGBM for {mkt}...")
        df = market_dfs[mkt]

        train_s, val_s, test_s, scaler, train_raw, val_raw, test_raw = \
            split_and_scale(df, LGBM_FEATURES)

        X_train = train_s[LGBM_FEATURES].values
        y_train = train_raw['log_price'].values


        X_val   = val_s[LGBM_FEATURES].values
        y_val   = val_raw['log_price'].values

        X_test  = test_s[LGBM_FEATURES].values
        y_test_log = test_raw['log_price'].values

        # ── Train ──────────────────────────────────────────────
        dtrain = lgb.Dataset(X_train, label=y_train)
        dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

        params = {
            'objective':        'huber',       # robust to ₹20k spikes
            'alpha':            0.9,            # huber threshold
            'metric':           'mae',
            'num_leaves':       128,
            'learning_rate':    0.5,
            'feature_fraction': 0.02,
            'bagging_fraction': 0.8,
            'bagging_freq':     5,
            'min_child_samples': 50,
            'reg_alpha':        0.8,
            'reg_lambda':       0.8,
            'verbose':          -1,
            'seed':             RANDOM_SEED,
        }

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,       # ← was 2000
            valid_sets=[dval],          # ← only val, not train (cleaner)
            valid_names=['val'],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=True),  # ← was 50
                lgb.log_evaluation(period=500),
            ]
        )

        # ── Evaluate ───────────────────────────────────────────
        val_pred_log  = model.predict(X_val)
        test_pred_log = model.predict(X_test)

        val_pred  = np.expm1(val_pred_log)
        test_pred = np.expm1(test_pred_log)
        y_val     = np.expm1(y_val)
        y_test    = np.expm1(test_raw['log_price'].values)

        print(f"  {mkt} results:")
        m_val  = compute_metrics(y_val,  val_pred,  'Validation')
        m_test = compute_metrics(y_test, test_pred, 'Test')
        all_metrics['lgbm'][mkt] = {'val': m_val, 'test': m_test}

        # ── Feature importance plot ────────────────────────────
        fig, ax = plt.subplots(figsize=(10, 7))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#1a1d27')
        imp = model.feature_importance(importance_type='gain')
        idx = np.argsort(imp)
        ax.barh([LGBM_FEATURES[i] for i in idx],
                imp[idx], color='#e74c3c', alpha=0.85)
        ax.set_title(f'LightGBM Feature Importance — {mkt}',
                     color='white', fontsize=12)
        ax.tick_params(colors='#aaa')
        plt.tight_layout()
        plt.savefig(f'outputs/lgbm_importance_{mkt}.png',
                    dpi=130, bbox_inches='tight', facecolor='#0f1117')
        plt.close()

        # ── Prediction plot ────────────────────────────────────
        plot_predictions(y_test, test_pred,
                         f'LightGBM {mkt}',
                         f'outputs/lgbm_pred_{mkt}.png')

        # ── Save model + scaler ────────────────────────────────
        model.save_model(f'outputs/lgbm_{mkt}.txt')
        with open(f'outputs/lgbm_scaler_{mkt}.pkl', 'wb') as f:
            pickle.dump(scaler, f)

        print(f"  Saved: outputs/lgbm_{mkt}.txt")


MODEL 1: LightGBM — 24h-Ahead Price Forecasting

  Training LightGBM for GDAM...
    Train: 103,393 rows  |  Val: 26,208  |  Test: 14,591
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[344]	val's l1: 0.155151
  GDAM results:
    Validation                      MAE=639.4  RMSE=982.7  MAPE=18.82%
    Test                            MAE=410.5  RMSE=703.9  MAPE=12.14%
  Saved: outputs/lgbm_GDAM.txt

  Training LightGBM for DAM...
    Train: 103,393 rows  |  Val: 26,208  |  Test: 14,591
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[319]	val's l1: 0.177489
  DAM results:
    Validation                      MAE=612.9  RMSE=1003.2  MAPE=23.45%
    Test                            MAE=413.8  RMSE=720.2  MAPE=17.87%
  Saved: outputs/lgbm_DAM.txt

  Training LightGBM for RTM...
    Train: 102,645 rows  |  Val: 26,208  |  Test: 14,591
Training until validation scores don't improve for 100 rounds


In [ ]:
import os
from google.colab import files

# 1. Zip the 'outputs' folder
# The syntax is: !zip -r [target_zip_name] [source_folder_path]
!zip -r /content/outputs.zip /content/outputs

# 2. Download the file to your local machine
files.download('/content/outputs.zip')

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/lgbm_scaler_DAM.pkl (deflated 18%)
  adding: content/outputs/lgbm_pred_RTM.png (deflated 5%)
  adding: content/outputs/lgbm_importance_DAM.png (deflated 17%)
  adding: content/outputs/lgbm_scaler_RTM.pkl (deflated 18%)
  adding: content/outputs/lgbm_importance_GDAM.png (deflated 16%)
  adding: content/outputs/lgbm_GDAM.txt (deflated 64%)
  adding: content/outputs/lgbm_RTM.txt (deflated 64%)
  adding: content/outputs/lgbm_importance_RTM.png (deflated 17%)
  adding: content/outputs/lgbm_DAM.txt (deflated 64%)
  adding: content/outputs/lgbm_scaler_GDAM.pkl (deflated 18%)
  adding: content/outputs/lgbm_pred_GDAM.png (deflated 4%)
  adding: content/outputs/lgbm_pred_DAM.png (deflated 5%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>